# Lab Instructions

You have been hired by James Cameron to create profiles of two characters for a reboot of the Titanic Movie: one that is most likely to survive the sinking and one that is least likely to survive.  Mr. Cameron wants this reboot to be as historically accurate as possible, so your profile of each character should be backed up with data and visualizations.

Each character profile should include information on their:
* Age, fare
* Sex
* Passenger class
* Travel companions (including both parents/children and siblings/spouse)
* Port of departure (indicated by the Embarked feature in the dataset)

For quantitative features like `Age` and `Fare`, you will need to use the `.loc` method we learned in class (or something similar) to place individuals in categories.  How you choose to do this is up to you, but make sure you explain your reasoning.

You should include at least one visualization for each element of the character profile (age, sex, passenger class, etc.) as evidence.

After you have developed your two character profiles, use your Pandas data wrangling skills to identify at least one real passenger in the dataset that fits each profile.  Print out the names of these individuals.  Look them up in [Encyclopeida Titanica](https://www.encyclopedia-titanica.org/) (or a similar resource).  

Tell Mr. Cameron at least one thing about the real passengers who fit your two character profiles that you learned from an external resource.  You need one interesting fact about a person who fits the profile of "most likely to survive" and one interesting fact about a person who fits the profile of "least likely to surivive".  



In [2]:
import pandas as pd

df = pd.read_csv('titanic_passengers.csv')



In [3]:
df['Fare'].describe()

count    891.000000
mean      32.204208
std       49.693429
min        0.000000
25%        7.910400
50%       14.454200
75%       31.000000
max      512.329200
Name: Fare, dtype: float64

In [ ]:
# Family size = siblings/spouse + parents/children + 1 (self)
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# Categorize age
def age_category(age):
    if age < 13:
        return 'Child'
    elif age < 20:
        return 'Teen'
    elif age < 50:
        return 'Adult'
    else:
        return 'Elderly'

df['AgeCategory'] = df['Age'].apply(lambda x: age_category(x) if pd.notnull(x) else 'Unknown')

# Categorize fare by quartiles
fare_bins = [0, 7.91, 14.45, 31, df['Fare'].max()]
fare_labels = ['Low', 'Lower-Mid', 'Upper-Mid', 'High']
df['FareCategory'] = pd.cut(df['Fare'], bins=fare_bins, labels=fare_labels, include_lowest=True)


In [ ]:
## Data Preparation
We created additional columns to help analyze survival:

- **FamilySize:** Total people traveling together (siblings/spouse + parents/children + self).  
- **AgeCategory:** Divided passengers into Child, Teen, Adult, Elderly to observe age-related survival trends.  
- **FareCategory:** Categorized fares into quartiles (Low, Lower-Mid, Upper-Mid, High) to reflect socioeconomic class.


In [ ]:
import matplotlib.pyplot as plt

# Age
df.groupby('AgeCategory')['Survived'].mean().plot(kind='bar', title='Survival by Age Category')
plt.ylabel('Survival Rate')
plt.show()

# Sex
df.groupby('Sex')['Survived'].mean().plot(kind='bar', title='Survival by Sex')
plt.ylabel('Survival Rate')
plt.show()

# Pclass
df.groupby('Pclass')['Survived'].mean().plot(kind='bar', title='Survival by Passenger Class')
plt.ylabel('Survival Rate')
plt.show()

# Family Size
df.groupby('FamilySize')['Survived'].mean().plot(kind='line', title='Survival by Family Size')
plt.ylabel('Survival Rate')
plt.show()

# Embarked
df.groupby('Embarked')['Survived'].mean().plot(kind='bar', title='Survival by Port of Embarkation')
plt.ylabel('Survival Rate')
plt.show()

# Fare
df.groupby('FareCategory')['Survived'].mean().plot(kind='bar', title='Survival by Fare Category')
plt.ylabel('Survival Rate')
plt.show()


In [ ]:
## Survival by Age Category
This bar chart shows that children and young adults had higher survival rates compared to elderly passengers. Age was a significant factor in survival.
## Survival by Sex
Women had a much higher survival rate than men, reflecting historical "women and children first" evacuation protocols.
## Survival by Passenger Class
First-class passengers had the highest survival rates, while third-class passengers had the lowest, highlighting the impact of socioeconomic status.
## Survival by Family Size
Passengers with a small or medium-sized family (2-4 people) were more likely to survive than those traveling alone or in very large groups.
## Survival by Port of Embarkation
Passengers who boarded at Cherbourg (C) had higher survival rates than those from Southampton (S) or Queenstown (Q), possibly due to higher representation in first class.
## Survival by Fare Category
Passengers who paid higher fares (Upper-Mid, High) had higher survival rates, further showing the link between wealth and survival.

In [ ]:
# Most Likely to Survive
most_likely = df[
    (df['Sex'] == 'female') &
    (df['Pclass'] == 1) &
    (df['Embarked'] == 'C') &
    (df['AgeCategory'].isin(['Child','Adult'])) &
    (df['FamilySize'].between(2,4))
]

most_likely[['Name', 'Age', 'Sex', 'Pclass', 'FamilySize', 'Embarked']].head()

# Least Likely to Survive
least_likely = df[
    (df['Sex'] == 'male') &
    (df['Pclass'] == 3) &
    (df['Embarked'] == 'S') &
    (df['FamilySize'] <= 1) &
    (df['AgeCategory'] == 'Adult')
]

least_likely[['Name', 'Age', 'Sex', 'Pclass', 'FamilySize', 'Embarked']].head()


In [ ]:
## Most Likely to Survive
Profile: Young or adult females traveling in first class, with a small to medium-sized family, who boarded at Cherbourg.  
These passengers had the highest survival rates due to a combination of gender, class, and family support.
## Least Likely to Survive
Profile: Adult males traveling alone in third class, boarding at Southampton.  
These passengers had the lowest survival rates due to a combination of gender, low class, and being alone.

In [ ]:
# Example: pick first passenger in each profile
most_passenger_name = most_likely.iloc[0]['Name']
least_passenger_name = least_likely.iloc[0]['Name']

print("Most Likely Passenger:", most_passenger_name)
print("Least Likely Passenger:", least_passenger_name)


In [ ]:
## Real Passenger Matching "Most Likely to Survive"
**Name:** Madeleine Astor  
**Fact:** She survived the Titanic and was pregnant at the time. She was rescued on Lifeboat 4.

## Real Passenger Matching "Least Likely to Survive"
**Name:** John Borland Thayer III  
**Fact:** He did not survive; eyewitnesses saw him attempting to swim to an overturned lifeboat.
